# Library

In [ ]:
import pandas as pd
import numpy as np
import os
import scipy.io
from datetime import datetime
from matplotlib import pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import sklearn.model_selection
import ast  # if values are stored as stringified lists
import seaborn as sns
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import accuracy_score, precision_score, f1_score, classification_report, confusion_matrix, roc_curve, roc_auc_score

# Path

In [ ]:
root_path = "/home/anphan/DigitalSelfHarm"
raw_data_path = os.path.join(root_path, "data/raw_data_sample.csv")
save_path = os.path.join(root_path,"data")
os.makedirs(save_path, exist_ok=True)
description_path = os.path.join(root_path,"data/data_description.csv")
log_path = os.path.join(save_path, "eda_preparation_log.txt")

# Data Description

In [ ]:
raw_df = pd.read_csv(raw_data_path)
target_cols = ['FL708', 'FL709']


print("# Rows =",len(raw_df))
print("# Cols =",len(raw_df.columns))
# print(raw_df.isnull().agg(['sum', lambda x: x.mean() * 100]).T.query("sum > 0"))

# Load the column info file
col_info = pd.read_csv(description_path)   # col_name, meaning, factor

# Compute missing summary
missing = raw_df.isnull().sum()
missing = missing[missing > 0]

In [ ]:
raw_df.shape

In [ ]:


with open(log_path, "w") as log_file:
    log_file.write(f"Data loaded at {datetime.now()}\n")
    log_file.write(f"Number of columns: {len(raw_df.columns)}\n")
    log_file.write(f"Data shape: {raw_df.shape}\n")
    log_file.write(f"target_cols = ['FL708', 'FL709']\n")


# Print with meanings
for col in missing.index:
    meaning = col_info.loc[col_info["col_name"] == col, "meaning"].values
    meaning = meaning[0] if len(meaning) > 0 else "N/A"
    print(f"{col}: missing {missing[col]} — meaning: {meaning}")
    with open(log_path, "a") as log_file:
        log_file.write(f"{col}: missing {missing[col]} — meaning: {meaning}\n")




In [ ]:

print("Target columns = \n FL708: NO OCCASIONS (12 MONTHS): DIGITAL SELF HARM? \n FL709: NO OCCASIONS (30 DAYS): DIGITAL SELF HARM?")
print("Value counts for target columns:")
with open(log_path, "a") as log_file:
    log_file.write(f"Target columns = \n FL708: NO OCCASIONS (12 MONTHS): DIGITAL SELF HARM? \n FL709: NO OCCASIONS (30 DAYS): DIGITAL SELF HARM?\n")
    
for col in target_cols: 
    print(f"Column: {col}")
    print(raw_df[col].value_counts())
    with open(log_path, "a") as log_file:
        log_file.write(f"Column: {col}\n")
        log_file.write(f"{raw_df[col].value_counts()}\n")
                       

In [ ]:
import pandas as pd
import numpy as np

def create_column_summary_with_missing_handling(df, col_info_path=None, col_info_df=None):
    """
    Analyzes dataframe where missing values are marked as strings like 'missing', 'NA', etc.
    Returns clean summary with meanings and proper missing value treatment.
    """
    # Define common "missing" string indicators
    missing_indicators = ['missing', 'Missing', 'MISSING', 'NA', 'N/A', 'na', 'n/a', '', ' ', 'null', 'NULL', 'None']
    
    if col_info_df is None and col_info_path is not None:
        col_info_df = pd.read_csv(col_info_path)
    elif col_info_df is None:
        col_info_df = pd.DataFrame({'col_name': df.columns.tolist(), 'meaning': 'N/A', 'factor': 'N/A'})
    
    col_info_df['col_name'] = col_info_df['col_name'].astype(str)
    df_columns_str = [str(c) for c in df.columns]
    
    summary_rows = []
    detailed_value_counts = []
    
    df_clean = df.copy()
    
    for col in df_clean.columns:
        col_str = str(col)
        
        before_replace = df_clean[col].isin(missing_indicators).sum()
        df_clean[col] = df_clean[col].replace(missing_indicators, np.nan)
        actual_missing = df_clean[col].isnull().sum()
        
        unique_count = df_clean[col].nunique(dropna=True)
        total_rows = len(df_clean)
        
        meaning_row = col_info_df[col_info_df['col_name'] == col_str]
        meaning = meaning_row['meaning'].iloc[0] if len(meaning_row) > 0 else "N/A"
        factor = meaning_row['factor'].iloc[0] if len(meaning_row) > 0 and 'factor' in col_info_df.columns else "N/A"
        
        vc_original = df[col].astype(str).value_counts(dropna=False)
        vc_pct = df[col].astype(str).value_counts(normalize=True, dropna=False) * 100
        
        vc_df = pd.DataFrame({
            'column': col_str,
            'value': vc_original.index,
            'count': vc_original.values,
            'percentage': vc_pct.values.round(2)
        }).sort_values(by='count', ascending=False)
        detailed_value_counts.append(vc_df)
        
        top5 = vc_original
        top5_str = " | ".join([f"{k}: {v}" for k, v in top5.items()])
        
        summary_rows.append({
            'column': col_str,
            'meaning': meaning,
            'factor': factor,
            'dtype': str(df[col].dtype),
            'missing_count': int(actual_missing),
            'missing_pct': round(actual_missing / total_rows * 100, 2),
            'unique_values': unique_count,
            'top_values': top5_str
        })
    
    summary_df = pd.DataFrame(summary_rows)
    detailed_df = pd.concat(detailed_value_counts, ignore_index=True)
    
    summary_cols = ['column', 'meaning', 'factor', 'dtype', 
                    'missing_count', 'missing_pct', 'unique_values', 'top_values']
    summary_df = summary_df[summary_cols]
    
    return summary_df, detailed_df



In [ ]:

summary_df, detailed_value_counts_df = create_column_summary_with_missing_handling(
    df=raw_df,
    col_info_path=description_path
)

print("Summary with meanings:")
print(summary_df)

print("\nDetailed value counts (shows 'missing' as a value):")
print(detailed_value_counts_df.head(15))

summary_df.to_csv(os.path.join(save_path, "column_summary_with_meaning.csv"), index=False)
detailed_value_counts_df.to_csv(os.path.join(save_path, "detailed_distributions_with_missing.csv"), index=False)

# Data Cleaning and Pre-processing (Encoding)

In [ ]:
df_processed = raw_df.copy()
df_processed = df_processed.drop(['ID','hsms', 'COUNTY', 'SchoolName', 'ZIP', 'stwtpop', 'stwtsamp','SCHOOLID', 'CLASSID'], axis = 1) 

step_counter = 0
with open(log_path, "a") as log_file:
    log_file.write(f"============ Preprocessing Steps ============\n")
    log_file.write(f"Initial shape: {df_processed.shape}\n")
    log_file.write(f"Step {step_counter}: Dropped identifier columns. New shape: {df_processed.shape}\n")
    log_file.write(f"identifier columns = ['ID','hsms', 'COUNTY', 'SchoolName', 'ZIP', 'stwtpop', 'stwtsamp', 'SCHOOLID', 'CLASSID'] \n")
    log_file.write(f"New shape: {df_processed.shape}\n")
step_counter += 1



In [ ]:
# Fill specific columns' missing values with 0

cols_to_fill = ['Q738', 
                'Q58A', 
                'Q58B', 
                'Q58C', 
                'FL700', 
                'FL701', 
                'FL68', 
                'U7']


df_processed[cols_to_fill] = df_processed[cols_to_fill].fillna(0)


with open(log_path, "a") as log_file:
    log_file.write(f"Step {step_counter}: Replace missing values in {cols_to_fill} with 0.\n")
print(df_processed.shape)
step_counter += 1

In [ ]:
def clean_age(value):
    if isinstance(value, str):
        value = value.strip().lower()
        if 'or older' in value:
            return int(value.split()[0])  # e.g., '19 or older' → 19
        elif value.isdigit():
            return int(value)
    return np.nan

df_processed['D1'] = df_processed['D1'].apply(clean_age)

with open(log_path, "a") as log_file:
    log_file.write(f"Step {step_counter}: Clean D1 column to extract age as integer, '19 or older' to 19. \n")
print(df_processed.shape)
step_counter += 1


In [ ]:
def clean_grade(value):
    if isinstance(value, str) and value.endswith('th'):
        # return value
        return int(value.replace('th', ''))
    return np.nan  # handle unexpected cases

df_processed['D2'] = df_processed['D2'].apply(clean_grade)

with open(log_path, "a") as log_file:
    log_file.write(f"Step {step_counter}: Clean D2 column to extract grade as integer. \n")
print(df_processed.shape)
step_counter += 1


In [ ]:
nominal_cols = ['D8','D3']
for col in nominal_cols:
    print(df_processed[col].value_counts())

df_processed = pd.get_dummies(df_processed, columns=nominal_cols, dummy_na=True)


with open(log_path, "a") as log_file:
    log_file.write(f"Step {step_counter}: One-hot encode nominal columns {nominal_cols} with dummy_na=True.\n")
    log_file.write(f"New shape: {df_processed.shape}\n")

print(df_processed.shape)
step_counter += 1


In [ ]:
binary_cols1 = ['D5A', 'D5B', 'D5C', 'D5D', 'D5E', 'D5F', 'D5G', 'D5H', 'D5I', 'D5J', 'D5K', 'D5L', 'D5M', 'D5N', 'D5O', 'D5P']

with open(log_path, "a") as log_file:
    log_file.write(f"Step {step_counter}: Encode binary columns {binary_cols1} with 1/0 ('Yes': 1, 'No': 0).\n")

df_processed[binary_cols1] = df_processed[binary_cols1].replace({'Yes': 1, 'No': 0})

print(df_processed.shape)
step_counter += 1


In [ ]:
recode_map = {    
    'Does not apply': 0,
    "Don't know" : 0,
    'Completed grade school or less' : 1, 
    'Some high school': 2,
    'Completed high school' : 3, 
    'Some college': 4, 
    'Completed college': 5,
    'Graduate or professional school after college': 6,
    "Missing": np.nan,
}
parent_schooling = ['D9', 'D10']
for col in parent_schooling:
    print(df_processed[col].value_counts())
    # df_processed[col] = df_processed[col].replace({'Missing': "Don't know"})
    # df_processed[col] = df_processed[col].replace(recode_map)

    df_processed[col] = df_processed[col].map(recode_map)


with open(log_path, "a") as log_file:
    log_file.write(f"Step {step_counter}: Encode parent schooling columns {parent_schooling} with recode_map:\n {recode_map} \n")
    log_file.write(f"New shape: {df_processed.shape}\n")

print(df_processed.shape)
step_counter += 1
df_processed['D10'].value_counts()


In [ ]:
living_map = {
    "On a farm": 'farm',
    "In the country, not on a farm": 'country',
    "In a city, town or suburb": 'city',
    "Missing": 'Missing'
}
print(df_processed['D11'].value_counts())

df_processed['D11'] = df_processed['D11'].map(living_map)

df_processed = pd.get_dummies(df_processed, columns=['D11'], prefix='Living', dummy_na=False)
with open(log_path, "a") as log_file:
    log_file.write(f"Step {step_counter}: Encode D11 column with living_map and create dummies. \n")
    log_file.write(f"New shape: {df_processed.shape}\n")

print(df_processed.shape)
step_counter += 1
# df_processed['D11'].value_counts()


In [ ]:
# Q13
recode_map = {    
    "Mostly A's" : 1,
    "Mostly B's" : 2,
    "Mostly C's" : 3,
    "Mostly D's" : 4,
    "Mostly F's" : 5,
    "Missing": np.nan,
}
study_result = ['Q13']
for col in study_result:
    print(df_processed[col].value_counts())
    df_processed[col] = df_processed[col].map(recode_map)


    #---------------------------------
    # df_processed[col] = df_processed[col].replace(recode_map)


with open(log_path, "a") as log_file:
    log_file.write(f"Step {step_counter}: Encode study result column Q13 with recode_map: \n {recode_map} \n")
print(df_processed.shape)
step_counter += 1

In [ ]:
df_processed['Q738'] = df_processed['Q738'].replace({'5-Apr': '4-5', '10-Jun': '6-10'})

recode_map = {    
    "0" : 0,
    "1" : 1,
    "2" : 2,
    "3" : 3,
    "4-5" : 4,
    "6-10" : 5,
    "11 or more" : 6,
    "Missing": np.nan,

}
school_missed = ['Q738']
for col in school_missed:
    print(df_processed[col].value_counts())
    df_processed[col] = df_processed[col].astype(str)
    # df_processed[col] = df_processed[col].replace(recode_map)

    df_processed[col] = df_processed[col].map(recode_map)




with open(log_path, "a") as log_file:
    log_file.write(f"Step {step_counter}:  Encode school_missed column Q738 with recode_map: \n {recode_map} \n")
print(df_processed.shape)
step_counter += 1

In [ ]:
yes_no_columns = ['FL48b','Q106', 'Q110', 'Q77', 'FL508']

recode_map = {    
    'No': 0, 'Yes': 1,
    "Missing": np.nan,
    "missing": np.nan,
}

for col in yes_no_columns:
    # print(df_processed[col].value_counts())
    df_processed[col] = df_processed[col].map(recode_map)

with open(log_path, "a") as log_file:
    log_file.write(f"Step {step_counter}: Normalize yes/no columns {yes_no_columns} with recode_map: \n {recode_map}\n")

print(df_processed.shape)
step_counter += 1

In [ ]:
# Fill bullied columns' '99' with 'Missing'
bullied_col = ['FL60X', 'FL61X', 'FL62X', 'FL63X', 'FL64X', 'FL65X']
df_processed[bullied_col] = df_processed[bullied_col].replace({'99': 'Missing'})

recode_map = {    
    "Never" : 0,
    "Once or twice" : 1,
    "A few times" : 2,
    "Many times" : 3,
    "Every day" : 4,
    "Missing": np.nan,
    "missing": np.nan,
}
for col in bullied_col:
    print(df_processed[col].value_counts())
    df_processed[col] = df_processed[col].map(recode_map)


with open(log_path, "a") as log_file:
    log_file.write(f"Step {step_counter}: Replace bullied columns' '99' with 'Missing'.\n")
    log_file.write(f"Step {step_counter}: Encode bullied columns {bullied_col} with recode_map: \n {recode_map} \n")
print(df_processed.shape)
step_counter += 1

In [ ]:

yes_no_columns = ['Q14','Q2891','Q15','Q2057','Q17','Q18','Q21','Q731','Q23','Q3668',
                'FL6','FL7', 'FL8', 'FL9',
                'Q27', 'Q29',
                'Q103A', 'Q103B', 'Q103C', 'Q103D', 
                'Q107',
                'Q76', 
                'Q2909', 'Q79', 'Q2911', 'Q82', 'Q83', 'Q84', 'Q85', 
                'Q89', 'Q93', 'Q94', 'Q96', 'Q99', 'Q78', 'Q2910', 'Q80']

recode_map = {    
    'NO!': 0, 'no': 1, 'yes': 2,'YES!': 3,
    "Missing": np.nan,
}

for col in yes_no_columns:
    # print(df_processed[col].value_counts())
    df_processed[col] = df_processed[col].map(recode_map)

with open(log_path, "a") as log_file:
    log_file.write(f"Step {step_counter}: Normalize yes/no columns {yes_no_columns} to 0/1/2/3/-1 ('NO!': 0, 'no': 1, 'yes': 2,'YES!': 3, 'Missing': -1, 'missing': -1).\n")

print(df_processed.shape)
step_counter += 1


In [ ]:
school_enjoy = ['Q3681', 'Q3684','Q3685','Q3686']
recode_map = {    
    "Never": 5,
    "Seldom":4,
    "Sometimes" : 3,
    "Often": 2,
    "Almost always":1,
    "Always": 0,
    "Missing": np.nan,
}
for col in school_enjoy:
    print(df_processed[col].value_counts())
    df_processed[col] = df_processed[col].map(recode_map)

with open(log_path, "a") as log_file:
    log_file.write(f"Step {step_counter}: Encode school_enjoy columns {school_enjoy} with recode_map: \n {recode_map} \n")
print(df_processed.shape)
step_counter += 1

In [ ]:
school_enjoy = ['Q3682']
recode_map = {    
    "Very dull": 4,
    "Slightly dull":3,
    "Quite interesting" : 2,
    "Fairly interesting": 1,
    "Very interesting and stimulating":0,
    "Missing": np.nan,
}
for col in school_enjoy:
    print(df_processed[col].value_counts())
    df_processed[col] = df_processed[col].map(recode_map)

with open(log_path, "a") as log_file:
    log_file.write(f"Step {step_counter}: Encode school_enjoy columns {school_enjoy} with recode_map: \n {recode_map} \n")
print(df_processed.shape)
step_counter += 1

In [ ]:
school_enjoy = ['Q3683']
recode_map = {    
    "Not at all important": 4,
    "Slightly important":3,
    "Fairly important" : 2,
    "Quite important": 1,
    "Very important": 0,
    "Missing": np.nan,
}
for col in school_enjoy:
    print(df_processed[col].value_counts())
    df_processed[col] = df_processed[col].map(recode_map)

with open(log_path, "a") as log_file:
    log_file.write(f"Step {step_counter}: Encode school_enjoy columns {school_enjoy} with recode_map: \n {recode_map} \n")
print(df_processed.shape)
step_counter += 1

In [ ]:
bad_habit_age = ['Q60A','Q60B','FL702','FL703','Q60C','Q60D','Q60E','Q60F','Q60G','Q60H']
recode_map = {    
    "Never have": 0,
    "17 or Older": 1,
    "16": 2,
    "15": 3,
    "14": 4,
    "13": 5,
    "12": 6,
    "11": 7,
    "10 or Younger": 8,
    "Missing": np.nan,
}
for col in bad_habit_age:
    print(df_processed[col].value_counts())
    df_processed[col] = df_processed[col].map(recode_map)

with open(log_path, "a") as log_file:
    log_file.write(f"Step {step_counter}: Encode bad_habit_age columns {bad_habit_age} with recode_map: \n {recode_map} \n")
print(df_processed.shape)
step_counter += 1

In [ ]:

recode_map = {    
    "Very wrong": 0,
    "Wrong": 1,
    "A little bit wrong": 2,
    "Not wrong at all": 3,
    "Missing": np.nan,
}
bad_mindset = ['Q61A','Q61B','Q61C','Q61D','Q61E','Q67A','Q67B','Q67C','FL704','FL705','Q67D',
               'Q33A','Q33B','Q33C']
for col in bad_mindset:
    print(df_processed[col].value_counts())
    df_processed[col] = df_processed[col].map(recode_map)

friends_habit = ['Q59AX','Q59BX','Q59CX','FL706','FL707','Q59DX']
for col in friends_habit:
    print(df_processed[col].value_counts())
    df_processed[col] = df_processed[col].map(recode_map)

family_good = ['Q74AX','Q74B','Q74C','FL125']
for col in family_good:
    print(df_processed[col].value_counts())
    df_processed[col] = df_processed[col].map(recode_map)

with open(log_path, "a") as log_file:
    log_file.write(f"Step {step_counter}: Encode bad_mindset columns {bad_mindset}\n friends habit {friends_habit}\n family_good {family_good}\n with recode_map: \n {recode_map} \n")
print(df_processed.shape)
step_counter += 1

In [ ]:
binary_cols2 = ['FL40', 'FL41', 'FL42', 'FL43', 'FL44']
recode_map = {    
    'Participated': 1, 
    'No participation': 0,
    "Missing": np.nan,
}
for col in binary_cols2:
    print(df_processed[col].value_counts())
    df_processed[col] = df_processed[col].map(recode_map)

with open(log_path, "a") as log_file:
    log_file.write(f"Step {step_counter}: Encode binary columns {binary_cols2} with 1/0 ('Participated': 1, 'No participation': 0).\n")

print(df_processed.shape)
step_counter += 1

In [ ]:
# : HOW FREQUENTLY DO YOU ATTEND RELIGIOUS SERIVCES?
# Rarely: 2941 | Never: 2718 | About once a week or more: 2567 | 1-2 times a month: 1273 | Missing: 320

binary_cols2 = ['Q54']
recode_map = {    
    "Never": 0,
    "Rarely": 1,
    "1-2 times a month": 2,
    "About once a week or more": 3,
    "Missing": np.nan,
}
for col in binary_cols2:
    print(df_processed[col].value_counts())
    df_processed[col] = df_processed[col].map(recode_map)

with open(log_path, "a") as log_file:
    log_file.write(f"Step {step_counter}: Encode religious columns {binary_cols2} with map {recode_map}.\n")

print(df_processed.shape)
step_counter += 1

In [ ]:
awareness = ['Q3687','Q3679','Q3688X','FL710','FL711','Q3680','FL120','FL116']
recode_map = {    
    "Great risk": 0,
    "Moderate risk": 1,
    "Slight risk": 2,
    "No risk": 3,
    "Missing": np.nan,
}
for col in awareness:
    print(df_processed[col].value_counts())
    df_processed[col] = df_processed[col].map(recode_map)

with open(log_path, "a") as log_file:
    log_file.write(f"Step {step_counter}: Encode awareness columns {awareness} with recode_map: \n {recode_map} \n")
print(df_processed.shape)
step_counter += 1

In [ ]:
# EVER SMOKED CIGARETTES?
awareness = ['U3']
recode_map = {    
    "Never": 0,
    "Once or twice": 1,
    "Once in a while but not regularly": 2,
    "Regularly in the past": 3,
    "Regularly now": 4,
    "Missing": np.nan,
}
for col in awareness:
    print(df_processed[col].value_counts())
    df_processed[col] = df_processed[col].map(recode_map)


with open(log_path, "a") as log_file:
    log_file.write(f"Step {step_counter}: Encode SMOKED-CIGARETTES columns {awareness} with recode_map: \n {recode_map} \n")
print(df_processed.shape)
step_counter += 1

In [ ]:
# FREQUENCY (30 DAY): CIGARETTES
awareness = ['U4']
recode_map = {    
    "Not at all": 0,
    "Less than one cigarette per day": 1,
    "One to five cigarettes per day": 2,
    "About one-half pack per day": 3,
    "About one pack per day": 4,
    "About one and one-half packs per day": 5,
    "Two packs or more per day": 6,
    "Missing": np.nan,
}
for col in awareness:
    print(df_processed[col].value_counts())
    df_processed[col] = df_processed[col].map(recode_map)

with open(log_path, "a") as log_file:
    log_file.write(f"Step {step_counter}: Encode N.o CIGARETTES columns {awareness} with recode_map: \n {recode_map} \n")
print(df_processed.shape)
step_counter += 1

In [ ]:
#bad_habit_frequency
bad_habit_frequency = ['U7']
recode_map = {    
    "0": 0,
    "Once": 1,
    "Twice": 2,
    "3-5 times": 3,
    "6-9 times": 4,
    "10 or more times": 5,
    "Missing": np.nan,
}
for col in bad_habit_frequency:
    print(df_processed[col].value_counts())
    df_processed[col] = df_processed[col].astype(str)
    df_processed[col] = df_processed[col].map(recode_map)

with open(log_path, "a") as log_file:
    log_file.write(f"Step {step_counter}: Encode bad_habit_frequency columns {bad_habit_frequency} with recode_map: \n {recode_map} \n")
print(df_processed.shape)
step_counter += 1

In [ ]:
#bad_habit_frequency
bad_habit_frequency = ['U5','U6','FL10','FL11','FL712','FL713','FL714','FL715','U10',
             'U11','U16','U17','FL49','FL50','FL66','FL67','FL51','FL52','U30X',
             'U31X','FL46','FL47','FL55','FL56','FL24','FL25','U24','U25']
recode_map = {    
    "0 occasions": 0,
    "1-2 occasions": 1,
    "3-5 occasions": 2,
    "6-9 occasions": 3,
    "10-19 occasions": 4,
    "20-39 occasions": 5,
    "40 or more occasions": 6,
    "Missing": np.nan,
}
for col in bad_habit_frequency:
    print(df_processed[col].value_counts())
    df_processed[col] = df_processed[col].map(recode_map)

with open(log_path, "a") as log_file:
    log_file.write(f"Step {step_counter}: Encode bad_habit_frequency columns {bad_habit_frequency} with recode_map: \n {recode_map} \n")
print(df_processed.shape)
step_counter += 1

In [ ]:
# behavior


behavior = ['FL502','FL503','FL504','FL505','FL506','FL507']
recode_map = {    
    "Strongly disagree": 0,
    "Disagree": 1,
    "Agree": 2,
    "Strongly agree": 3,
    "Missing": np.nan,
}
for col in behavior:
    print(df_processed[col].value_counts())
    df_processed[col] = df_processed[col].map(recode_map)

with open(log_path, "a") as log_file:
    log_file.write(f"Step {step_counter}: Encode behavior columns {behavior} with recode_map: \n {recode_map} \n")
    log_file.write(f"New shape: {df_processed.shape}\n")

print(df_processed.shape)
step_counter += 1

In [ ]:
#self control


control = ['Q25','Q26','Q28','Q30','Q32']
recode_map = {    
    "Very Hard": 0,
    "Sort of Hard": 1,
    "Sort of Easy": 2,
    "Very Easy": 3,
    "Missing": np.nan,
}
for col in control:
    print(df_processed[col].value_counts())
    df_processed[col] = df_processed[col].map(recode_map)

with open(log_path, "a") as log_file:
    log_file.write(f"Step {step_counter}: Encode control columns {control} with recode_map: \n {recode_map} \n")
print(df_processed.shape)
step_counter += 1


In [ ]:
control = ['Q104','Q108']
recode_map = {    
    "Never": 0,
    "1 or 2 times": 1,
    "3 or 4 times": 2,
    "5 or 6 times": 3,
    "7 or more times": 4,
    "Missing": np.nan,
}
for col in control:
    print(df_processed[col].value_counts())
    df_processed[col] = df_processed[col].map(recode_map)

with open(log_path, "a") as log_file:
    log_file.write(f"Step {step_counter}: Encode change school/homme columns {control} with recode_map: \n {recode_map} \n")
print(df_processed.shape)
step_counter += 1


In [ ]:
control = ['Q66A','Q66B','Q66C','Q66D','Q66E','Q66F','Q66H','FL122','FL123','FL124']
recode_map = {    
    "Never": 0,
    "1 or 2 times": 1,
    "3 to 5 times": 2,
    "6 to 9 times": 3,
    "10 to 19 times": 4,
    "20 to 29 times": 5,
    "30 to 39 times": 6,
    "40+ times": 7,
    "Missing": np.nan,
}
for col in control:
    print(df_processed[col].value_counts())
    df_processed[col] = df_processed[col].map(recode_map)

with open(log_path, "a") as log_file:
    log_file.write(f"Step {step_counter}: Encode illegal_activity, aggression_violence, handgun, substance_use columns {control}with recode_map: \n {recode_map} \n")
print(df_processed.shape)
step_counter += 1


In [ ]:


family_good = ['Q91','Q86']
recode_map = {    
    "Never or almost never": 0,
    "Sometimes": 1,
    "Often": 2,
    "All the time": 3,
    "Missing": np.nan,
}
for col in family_good:
    print(df_processed[col].value_counts())
    df_processed[col] = df_processed[col].map(recode_map)

with open(log_path, "a") as log_file:
    log_file.write(f"Step {step_counter}: Encode family_good columns {family_good} with recode_map: \n {recode_map} \n")
print(df_processed.shape)
step_counter += 1


In [ ]:

# HOURS OF SLEEP ON AVERAGE SCHOOL NIGHT
sleep_duration = ['FL509']
recode_map = {    
    "8 hours": 8,
    "7 hours": 7,
    "6 hours": 6,
    "5 hours": 5,
    "4 hours or less": 4,
    "9 hours": 9,
    "10 hours or more": 10,
    "Missing": np.nan,
}


for col in sleep_duration:
    print(df_processed[col].value_counts())
    df_processed[col] = df_processed[col].map(recode_map)

with open(log_path, "a") as log_file:
    log_file.write(f"Step {step_counter}: Encode sleep_duration columns {sleep_duration} with recode_map: \n {recode_map} \n")
print(df_processed.shape)
step_counter += 1


In [ ]:

# HOURS PER WEEK HANGING WITH FRIENDS WITH NO ADULTS PRESENT
sleep_duration = ['FL510X']


recode_map = {
    "0 hours": 0,
    "1 to 2 hours": 1.5,
    "3 to 4 hours": 3.5,
    "5 to 6 hours": 5.5,
    "7 to 8 hours": 7.5,
    "9 to 10 hours": 9.5,
    "11 to 12 hours": 11.5,
    "13 to 14 hours": 13.5,
    "15 to 16 hours": 15.5,
    "17 to 18 hours": 17.5,
    "19 to 20 hours": 19.5,
    "21 to 22 hours": 21.5,
    "23 to 24 hours": 23.5,
    "More than 24 hours": 25,
    "Missing": np.nan
}

for col in sleep_duration:
    print(df_processed[col].value_counts())
    df_processed[col] = df_processed[col].map(recode_map)

with open(log_path, "a") as log_file:
    log_file.write(f"Step {step_counter}: Encode hangout columns {sleep_duration} with recode_map: \n {recode_map} \n")
print(df_processed.shape)
step_counter += 1


In [ ]:
missing_nominal_cols = ['Q13', 'Q738', 'Q58A', 'Q58B', 'Q58C', 'FL700', 'FL701', 'FL68']

for col in missing_nominal_cols:
    # print(df_processed[col].value_counts())
    df_processed[col] = df_processed[col].replace({'Missing': np.nan, 'missing': np.nan})

    
with open(log_path, "a") as log_file:
    log_file.write(f"Step {step_counter}: Replace 'Missing' and 'missing' in {missing_nominal_cols} with np.nan.\n")
print(df_processed.shape)
step_counter += 1

In [ ]:
is_missing = df_processed.isna()
df_processed['missing_count'] = is_missing.sum(axis=1)

print(f"Total Rows of raw data                       : {len(raw_df)}")
print(f"Total Rows after {step_counter-1} steps                    : {len(df_processed)}")

In [ ]:
df_processed.to_csv(os.path.join(save_path, "data_processed.csv"), index=False)